# Structured Outputs 🧱📋

For a long time, integrating LLMs into software pipelines felt like writing code that talks to an unpredictable human: you asked for JSON, and the model would often reply with:

"Sure! Here is your JSON object:" followed by markdown code blocks, trailing conversational text, or mismatched data types.

In production software, a trailing sentence or a missing comma crashes your API router (JSON.parse() failure). Structured Outputs solve this by enforcing rigid, schema-backed data contracts between your app and the LLM.

## Phase 1: The Problem with Legacy Text Parsing
When building traditional apps, your functions return typed objects or rigid JSON responses. Early prompt engineering tried to force this by adding instructions like: "Output valid JSON only."

Why did this fail in production?

**Conversational Filler:** Models are heavily aligned to be helpful human assistants, so they love adding polite intros and outros.

**Markdown Wrapping:** Models frequently wrap JSON inside json ...  tags, requiring brittle regex hacks in your backend code to strip them out.

**Type Drift:** A model might return a string "42" instead of an integer 42, breaking downstream database inserts or TypeScript interfaces.

## Phase 2: The Evolution of Enforcing Structure
Today, major LLM APIs and modern frameworks handle structured outputs through two primary architectural layers:

### 1. JSON Mode (Soft Constraint)
**How it works:** You pass a flag in the API request (e.g., response_format: { "type": "json_object" }) combined with a system prompt instructing the model to output JSON.

**Result:** It guarantees that the output string is syntactically valid JSON, but it does not guarantee that the JSON matches your specific schema (it might still miss fields or use wrong data types).

### 2. Native Structured Outputs / Constrained Decoding (Hard Constraint)
**How it works:** You pass a strict JSON Schema (or a Pydantic model in Python / Zod schema in TypeScript) directly to the API provider.

**Under the Hood (Constrained Decoding):** The inference engine actively masks token probabilities at every single generation step. If a token violates your schema definition (e.g., trying to write a string where an integer is required), that token has its probability set to 0.

**Result:** 100% guarantee that the returned payload matches your exact data contract. No regex stripping, no parsing errors.

## Phase 3: Developer Blueprint — Using Pydantic & OpenAI
In Python, the industry standard for defining data contracts is Pydantic. Here is how you bind a Pydantic schema directly to an LLM API call to get guaranteed typed objects back.

In [ ]:
from pydantic import BaseModel, Field
from openai import OpenAI

client = OpenAI()

# 1. Define your strict data contract using Pydantic
class BugReportAnalysis(BaseModel):
    issue_summary: str = Field(description="A brief 1-sentence summary of the bug.")
    severity_level: int = Field(description="Severity score from 1 (low) to 5 (critical).")
    affected_component: str = Field(description="The system component, e.g., 'Auth', 'Database', 'UI'.")
    tags: list[str] = Field(description="List of relevant tech stack tags.")

# 2. Make the API call using parse() with response_format
def extract_structured_bug(raw_log: str) -> BugReportAnalysis:
    completion = client.beta.chat.completions.parse(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Extract structured engineering data from raw application error logs."},
            {"role": "user", "content": raw_log},
        ],
        response_format=BugReportAnalysis, # Pass the Pydantic class directly!
    )
    
    # The SDK automatically validates and parses it into a strongly-typed Python object
    return completion.choices[0].message.parsed

# Test the function
log_data = "FATAL [AuthService]: NullPointerException during JWT token validation in OAuth callback handler."
result = extract_structured_bug(log_data)

print(type(result))          # Output: <class '__main__.BugReportAnalysis'>
print(result.severity_level) # Output: 5 (Guaranteed integer)
print(result.affected_component) # Output: Auth

## Phase 4: Why This Matters for Web & Software Developers
By implementing structured outputs, you bridge the gap between probabilistic AI and deterministic code:

**Type Safety:** Seamlessly integrate LLM responses into TypeScript interfaces, Pydantic classes, or database ORMs (like Prisma, SQLAlchemy, or Mongoose).

**Reliable Automation:** Build multi-agent systems where Agent A outputs structured data that Agent B consumes without breaking parsing logic.

**Elimination of Boilerplate:** Say goodbye to complex regex cleaners and custom string-parsing try/catch blocks.